# 04 — Render handyman CVs as PDFs into the Volume

Renders one PDF per handyman (10,000) from the structured CV in `synth_handymen_truth` and writes it to
`/Volumes/bootcamp_students/maintops/maintops_docs/handyman_cvs/cv_<user_id>.pdf` …, exactly the `cv_path` stored in `synth_handyman_details`.

To give the CVs some **variety** for `ai_parse_document`, three different layouts and six colour themes are used:
0. classic single column, 1. shaded sidebar + main column, 2. coloured banner header with a timeline table.

Rendering is spread over the cluster with `mapInPandas`. Re-running overwrites the same files.

In [11]:
%pip install reportlab

Note: you may need to restart the kernel to use updated packages.


In [12]:
try:
    dbutils.library.restartPython()
except NameError:  # running locally: no dbutils, nothing to restart
    pass

In [13]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = ['reportlab']   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [14]:
%run ./00_config

In [15]:
# Cell 1 — Volume + target folder
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
if IS_DATABRICKS:
    dbutils.fs.mkdirs(CV_DIR)
else:
    from databricks.sdk import WorkspaceClient
    WorkspaceClient().files.create_directory(CV_DIR)
print(f"CVs go to: {CV_DIR}")

I0921 17:54:46.824910  553141 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


CVs go to: /Volumes/bootcamp_students/maintops/maintops_docs/handyman_cvs


In [16]:
# Cell 2 — PDF renderer (self-contained: it is shipped to the workers)
THEMES = ["#1f4e79", "#2e7d32", "#8e2c2c", "#4a3f8f", "#00695c", "#37474f"]

def render_cv(cv, path, template, theme_index):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
    from xml.sax.saxutils import escape as esc

    accent = colors.HexColor(THEMES[theme_index % len(THEMES)])
    body   = ParagraphStyle("body", fontName="Helvetica", fontSize=9.5, leading=12.5)
    small  = ParagraphStyle("small", parent=body, fontSize=8.5, leading=11)
    h1     = ParagraphStyle("h1", fontName="Helvetica-Bold", fontSize=22, leading=26, textColor=accent)
    h1w    = ParagraphStyle("h1w", parent=h1, textColor=colors.white)
    sub    = ParagraphStyle("sub", fontName="Helvetica", fontSize=11.5, leading=14, textColor=colors.HexColor("#444444"))
    subw   = ParagraphStyle("subw", parent=sub, textColor=colors.white)
    h2     = ParagraphStyle("h2", fontName="Helvetica-Bold", fontSize=11, leading=14, textColor=accent, spaceBefore=9, spaceAfter=2)
    bold   = ParagraphStyle("bold", parent=body, fontName="Helvetica-Bold")
    P = lambda t, s=body: Paragraph(esc(str(t)), s)

    def experience(width_style=body):
        out = []
        for j in cv["jobs"]:
            out.append(Paragraph(f"<b>{esc(j['role'])}</b> - {esc(j['employer'])}, {esc(j['city'])}", width_style))
            out.append(Paragraph(f"<font color='#666666'>{j['from']} - {j['to']}</font>", small))
            out += [Paragraph(f"&bull; {esc(b)}", width_style) for b in j["bullets"]]
            out.append(Spacer(1, 4))
        return out

    def certs():     return [Paragraph(f"{c['year']} &nbsp; {esc(c['name'])}", small) for c in cv["certifications"]]
    def education(): return [Paragraph(f"{e['year']} &nbsp; {esc(e['name'])}", small) for e in cv["education"]]
    contact = [cv["email"], cv["phone"], cv["address"]]

    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=16*mm, rightMargin=16*mm, topMargin=14*mm, bottomMargin=14*mm,
                            title=f"CV - {cv['name']}", author=cv["name"])
    W = A4[0] - 32*mm
    flow = []

    if template == 0:                                   # classic
        flow += [P(cv["name"], h1), P(cv["headline"], sub), Spacer(1, 3), P("  |  ".join(contact), small),
                 HRFlowable(width="100%", thickness=1.2, color=accent, spaceBefore=6, spaceAfter=4),
                 P("Profile", h2), P(cv["summary"]),
                 P("Skills", h2), P(", ".join(cv["skills"])),
                 P("Professional experience", h2)] + experience() + \
                [P("Certifications", h2)] + certs() + [P("Education", h2)] + education() + \
                [P("Languages", h2), P(", ".join(cv["languages"]))] + ([P("Other", h2), P(", ".join(cv["extras"]))] if cv["extras"] else [])

    elif template == 1:                                 # sidebar
        side = [P(cv["name"], ParagraphStyle("n", parent=h1, fontSize=17, leading=20)), P(cv["headline"], small), Spacer(1, 6),
                P("Contact", h2)] + [P(c, small) for c in contact] + \
               [P("Specialisations", h2)] + [P(s.replace("_", " ").title(), small) for s in cv["specialisations"]] + \
               [P("Skills", h2)] + [P(f"- {s}", small) for s in cv["skills"]] + \
               [P("Languages", h2)] + [P(l, small) for l in cv["languages"]] + \
               ([P("Other", h2)] + [P(e, small) for e in cv["extras"]] if cv["extras"] else [])
        main = [P("Profile", h2), P(cv["summary"]), P("Professional experience", h2)] + experience() + \
               [P("Certifications", h2)] + certs() + [P("Education", h2)] + education()
        t = Table([[side, main]], colWidths=[W * .34, W * .66])
        t.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("BACKGROUND", (0, 0), (0, 0), colors.HexColor("#f0f2f5")),
                               ("LEFTPADDING", (0, 0), (0, 0), 8), ("RIGHTPADDING", (0, 0), (0, 0), 8), ("LEFTPADDING", (1, 0), (1, 0), 14)]))
        flow.append(t)

    else:                                               # banner + timeline table
        banner = Table([[P(cv["name"], h1w)], [P(cv["headline"], subw)], [P("  |  ".join(contact), ParagraphStyle("c", parent=small, textColor=colors.white))]], colWidths=[W])
        banner.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), accent), ("LEFTPADDING", (0, 0), (-1, -1), 10), ("TOPPADDING", (0, 0), (0, 0), 10), ("BOTTOMPADDING", (0, -1), (-1, -1), 10)]))
        rows = [[P(f"{j['from']} - {j['to']}", bold),
                 [Paragraph(f"<b>{esc(j['role'])}</b>", body), P(f"{j['employer']}, {j['city']}", small)] + [Paragraph(f"&bull; {esc(b)}", small) for b in j["bullets"]]] for j in cv["jobs"]]
        tl = Table(rows, colWidths=[W * .2, W * .8])
        tl.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("LINEBELOW", (0, 0), (-1, -1), .4, colors.HexColor("#cccccc")), ("BOTTOMPADDING", (0, 0), (-1, -1), 6)]))
        flow += [banner, Spacer(1, 8), P("Profile", h2), P(cv["summary"]), P("Experience", h2), tl,
                 P("Key skills", h2), P(" - ".join(cv["skills"])), P("Certifications", h2)] + certs() + \
                [P("Education", h2)] + education() + [P("Languages", h2), P(", ".join(cv["languages"]))]

    doc.build(flow)


def render_partition(batches):
    import json, pandas as pd
    for pdf in batches:
        out = []
        for r in pdf.itertuples():
            cv = json.loads(r.cv_json)
            try:
                try:
                    render_cv(cv, r.cv_path, int(r.user_id) % 3, int(r.user_id) // 3)
                except Exception:                       # e.g. sidebar layout too tall -> fall back to the classic layout
                    render_cv(cv, r.cv_path, 0, int(r.user_id) // 3)
                out.append((int(r.user_id), r.cv_path, True, ""))
            except Exception as e:
                out.append((int(r.user_id), r.cv_path, False, str(e)[:200]))
        yield pd.DataFrame(out, columns=["user_id", "cv_path", "ok", "error"])

In [17]:
# Cell 3 — Render one PDF per handyman
src = (spark.table(T_HANDYMAN_DETAILS).select("user_id", "cv_path")
            .join(spark.table(T_HANDYMEN_TRUTH).select("user_id", "cv_json"), "user_id")
            .repartition(64))

result = src.mapInPandas(render_partition, schema="user_id long, cv_path string, ok boolean, error string")
result.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("synth_cv_render_log"))

log = spark.table(fq("synth_cv_render_log"))
print(f"Rendered OK : {log.where('ok').count():,}")
print(f"Failed      : {log.where('NOT ok').count():,}")
display(log.where("NOT ok").limit(20))

Rendered OK : 10,000
Failed      : 0


,id,cv_path,ok,error


In [18]:
# Cell 4 — Check the Volume
import os
files = list_volume_files(CV_DIR, ".pdf")
print(f"{len(files):,} PDFs in {CV_DIR}")
print(files[:5])

10,000 PDFs in /Volumes/bootcamp_students/maintops/maintops_docs/handyman_cvs
['cv_00001.pdf', 'cv_00002.pdf', 'cv_00003.pdf', 'cv_00004.pdf', 'cv_00005.pdf']
